# analysis_D — Pandas 시계열

- 담당: (이름)  브랜치: `<본인 github id>`
- 규칙: 커밋 전 `Kernel → Restart & Clear Output`. 이 노트북은 1인 전용.


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
pf = pd.read_csv(ROOT/"data/portfolio.csv")
sp = pd.read_csv(ROOT/"data/stock_prices.csv")
pf.columns, sp.columns = pf.columns.str.strip().str.lower(), sp.columns.str.strip().str.lower()
sp["trade_date"] = pd.to_datetime(sp["trade_date"])
pf.head(), sp.head()

## 1. 일자별 총 평가금액  (SQL [B-1]/[B-2]와 동일)

In [ ]:
m = sp.merge(pf[["ticker","quantity"]], on="ticker")
m["eval"] = m["quantity"] * m["close_price"]
daily = m.groupby("trade_date", as_index=False)["eval"].sum().rename(columns={"eval":"total_eval"})
daily["diff_amt"] = daily["total_eval"].diff()
daily["diff_pct"] = (daily["total_eval"].pct_change() * 100).round(2)
daily

## 2. 변화가 가장 컸던 날 / 주간 집계 / 최대 낙폭

In [ ]:
biggest_up = daily.nlargest(5, "diff_pct")
biggest_down = daily.nsmallest(5, "diff_pct")
weekly = daily.set_index("trade_date")["total_eval"].resample("W").agg(["first","last","min","max"])
weekly["change_pct"] = ((weekly["last"]-weekly["first"])/weekly["first"]*100).round(2)
peak = daily["total_eval"].cummax()
daily["drawdown_pct"] = ((daily["total_eval"]-peak)/peak*100).round(2)
biggest_up, biggest_down, weekly

## 3. 시각화 (선택)

In [ ]:
ax = daily.plot(x="trade_date", y="total_eval", figsize=(10,4), title="Portfolio total value")

## 4. 시장지수 비교 (SQL [B-6]/[B-7]과 동일)

In [ ]:
mi = pd.read_csv(ROOT/"data/market_index.csv")
mi.columns = mi.columns.str.strip().str.lower()
mi["trade_date"] = pd.to_datetime(mi["trade_date"])
idx_name = sorted(mi["index_name"].unique())[0]          # SQL과 동일하게 첫 번째 지수
idx = mi[mi["index_name"] == idx_name][["trade_date","close_value"]]
vs = daily[["trade_date","total_eval"]].merge(idx, on="trade_date", how="inner")
vs["port_pct"]  = (vs["total_eval"].pct_change()*100).round(2)
vs["index_pct"] = (vs["close_value"].pct_change()*100).round(2)
vs["excess_pct"] = (vs["port_pct"] - vs["index_pct"]).round(2)
vs["rel_strength"] = ((vs["total_eval"]/vs["total_eval"].iloc[0]) / (vs["close_value"]/vs["close_value"].iloc[0])).round(4)
print("거래일 불일치:", len(set(daily.trade_date) ^ set(idx.trade_date)))
vs

In [ ]:
shock = vs.dropna().reindex(vs["index_pct"].abs().sort_values(ascending=False).index).head(5)
shock["direction"] = np.where((shock.index_pct>0)==(shock.port_pct>0), "same", "opposite")
shock[["trade_date","index_pct","port_pct","direction"]]

## 5. 초과수익 최대/최소 구간, 상대강도 추이

In [ ]:
best, worst = vs.nlargest(3,"excess_pct"), vs.nsmallest(3,"excess_pct")
ax = vs.plot(x="trade_date", y="rel_strength", figsize=(10,3), title=f"Portfolio / {idx_name} (relative strength)")
best, worst

## 6. 결과 내보내기

In [ ]:
out = ROOT/"outputs"; out.mkdir(exist_ok=True)
daily.to_csv(out/"pandas_D_daily_total.csv", index=False)
weekly.reset_index().to_csv(out/"pandas_D_weekly.csv", index=False)
vs.to_csv(out/"pandas_D_vs_index.csv", index=False)
shock.to_csv(out/"pandas_D_index_shock_days.csv", index=False)